# Mod 6 Code Assignment 22 — Streamlit “Serve the Model” (Penguins)
**Goal:** Build the **serving app only**. You will *not* train in this assignment. You’ll load a pre-baked pipeline (`penguin_model.pkl`) and its metadata (`penguin_meta.json`), create a clean Streamlit UI (sidebar + columns), and serve predictions.

---

## Instructor Guidance (Docs + Pseudocode)

**Docs**
- Streamlit Basics: https://docs.streamlit.io/
- Streamlit Caching: https://docs.streamlit.io/develop/concepts/architecture/caching
- Joblib load/dump: https://joblib.readthedocs.io/en/latest/persistence.html
- `predict_proba` (sklearn): https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

**Pseudocode Plan**
1) **Create project folder & venv** → install pinned `requirements.txt`.  
2) **Place baked files** (`penguin_model.pkl`, `penguin_meta.json`) in project root.  
3) **Make `app.py`**:  
   - `@st.cache_resource` → load model; `@st.cache_data` → load meta.  
   - Sidebar sliders for `bill_length_mm`, `flipper_length_mm` + a `selectbox` for image hint + **Predict** button.  
   - Main area uses `st.columns`: left shows inputs (`st.dataframe`), right shows prediction + class probabilities.  
   - Optional `st.image` placeholder for chosen species.  
4) **Run the app** → `streamlit run app.py`.  
5) **Test**: move sliders, click Predict, confirm probabilities sum ≈ 1.

### YOU DO:  Continue to Build Off Of the penguin model you saved as a .pkl file and the slider data you saved as a .json file to create a streamlit app

- You can add additional functionality/change the aesthetics if you like
- Remember to SAVE app changes and to kill the app in your terminal type CTRL + C 
- You need to drop your .pkl, .json, and requirements.txt files into the same folder as the app.py file so you can use them 


### Copy and paste the code below into an app.py file.  Fill in the nones

In [ ]:
# app.py
import streamlit as st
import joblib, json, numpy as np, pandas as pd

st.set_page_config(page_title=None, page_icon="🐧", layout="centered")
st.title("penguin vibes")

@st.cache_resource
def load_model():
    return joblib.load("penguin_model.pkl") #use joblin.load 

@st.cache_data
def load_meta():
    return json.load(open("penguin_meta.json")) #use json.load 

model = load_model() #call the function that loads the model
meta = load_meta()

st.caption(f"Model trained on TRAIN split; evaluated on TEST. Reported TEST accuracy: **{meta.get('test_accuracy', 'n/a')}**")

# --- Sidebar inputs ---
st.sidebar.header("Input Features")
bl = st.sidebar.slider(
    "Bill Length (mm)",
    float(meta['bill_length_mm'][0], float(meta['bill_length_mm'[1]])),
    float(meta['bill_length_mm']), step=0.1
)
fl = st.sidebar.slider(
    "Flipper Length (mm)",
    float(meta["flipper_length_mm"][0]), float(meta["flipper_length_mm"][1]),
    float(meta["flipper_length_default"]), step=1.0
)

species_hint = st.sidebar.selectbox("Show example image for:", meta["classes"], index=0)
go = st.sidebar.button("Predict")

# --- Two columns for outputs ---
col1, col2 = st.columns(2)

with col1:
    st.subheader("Your Inputs")
    st.dataframe(pd.DataFrame([{
        "Bill Length (mm)": bl, #name of input columns as a str
        "Flipper Length (mm)": fl
    }]), use_container_width=True)

with col2:
    st.subheader("Prediction")
    if go:
        X = np.array([[bl, fl]])
        pred = model.predict(X)
        proba = model.predict_proba[X]
        st.success(f"Predicted species: **{pred}**")
        st.caption("Class probabilities:")
        for c, p in zip(model.classes_, proba):
            st.write(f"- {c}: {p:.3f}")
    else:
        st.info("Adjust sliders and click **Predict** in the sidebar.")

st.divider()
st.subheader("Example Species Image")
#change the placeholder image if you like 
st.image(
    f"https://placehold.co/600x300?text={species_hint}",
    caption=f"Placeholder image: {species_hint}",
    use_column_width=True
)
